# How to work with the models 

The loading of the model is in *model.py*. The files *arch.py, cdpath_models.py, resnet_ret.py and vision_transformer.py* contain architectures specific to some models.

In [1]:
from wp1.models.model import *
from PIL import Image 
from pathlib import Path
from wp1.utils.inference_utils import get_weight
from wp1.dataset.dataset import open_and_convert_image

/home/axelle/Documents/Doctorat/Main research/WP1 Patch retrieval/code/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/axelle/Documents/Doctorat/Main research/WP1 Patch retrieval/code/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


# Select your model #
The supported models are listed below. The weight argument links to where to obtain the weights. Some models do not require downloading and are directly accessible through the code (pytorch¹ or huggingface hub²). Some may require to request permission³: 
- model_name: "resnet", num_features: 128, weight: /¹, paper: [*Deep residual learning for Image recognition*](https://arxiv.org/abs/1512.03385)
- model_name:  "deit", num_features: 128, weight: /¹, paper: [*Training data-efficient image transformers & distillation through attention*](https://arxiv.org/abs/2012.12877)
- model_name: "dino"_+{pre/scratch/fine}, num_features: 384, weight: [ImageNet pretrained](https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_linearweights.pth) from [official github](https://github.com/facebookresearch/dino), paper: [*Emerging Properties in Self-Supervised Vision Transformers*](https://arxiv.org/abs/2104.14294)
- model_name: "dinov3", num_features: 768, weight: /
- model_name: "ibot"_+{pre/scratch/fine}, num_features: 384, weight: [ImageNet pretrained](https://lf3-nlp-opensource.bytetos.com/obj/nlp-opensource/archive/2022/ibot/vits_16/checkpoint_teacher.pth) from [official github](https://github.com/bytedance/ibot), paper: [*iBOT: Image BERT Pre-Training with Online Tokenizer*](https://arxiv.org/abs/2111.07832)
- model_name: "byol"_+{pre/scratch/fine}, num_features: 384, weight: [ImageNet pretrained](https://lightly-ssl-checkpoints.s3.amazonaws.com/imagenet_resnet50_byol_2024-02-14_16-10-09/pretrain/version_0/checkpoints/epoch%3D99-step%3D500400.ckpt) from [lightly](https://github.com/lightly-ai/lightly?tab=readme-ov-file), paper: [*Bootstrap your own latent: A new approach to self-supervised Learning*](https://arxiv.org/abs/2006.07733)
- model_name: "cdpath", num_features: 512, weight: [Camelyon pretrained](https://nextcloud.centralesupelec.fr/s/S4AZtXSw8Nj2p32/download/CAMELYON17_XY.ckpt) from [official github](https://github.com/jcboyd/cdpath21-gan), paper:  [*Self-Supervised Representation Learning using Visual Field Expansion on Digital Pathology*](https://openaccess.thecvf.com/content/ICCV2021W/CDPath/papers/Boyd_Self-Supervised_Representation_Learning_Using_Visual_Field_Expansion_on_Digital_Pathology_ICCVW_2021_paper.pdf)
- model_name: "ctranspath, num_features: 768, weight: [TCGA pretrained](https://drive.google.com/file/d/1DoDx_70_TLj98gTf6YTXnu4tFhsFocDX/view?usp=sharing) from [official github](https://github.com/Xiyue-Wang/TransPath), paper: [*Transformer-based unsupervised contrastive learning for histopathological image classification*](https://www.sciencedirect.com/science/article/abs/pii/S1361841522002043)
- model_name: "phikon_v1", num_features: 768, weight: [/²](https://huggingface.co/owkin/phikon), paper: [*Scaling Self-Supervised Learning for Histopathology with Masked Image Modeling*](https://www.medrxiv.org/content/10.1101/2023.07.21.23292757v2)
- model_name: "phikon_v2", num_features: 1024, weight: [/²](https://huggingface.co/owkin/phikon-v2), paper: [*Phikon-v2, A large and public feature extractor for biomarker prediction*](https://arxiv.org/abs/2409.09173)
- model_name: "uni_v1", num_features: 1024, weight: [/²,³](https://github.com/mahmoodlab/UNI), paper: [*Towards a General-Purpose Foundation Model for Computational Pathology*](https://www.nature.com/articles/s41591-024-02857-3)
- model_name: "uni_v2", num_features: 1536, weight: [/²,³](https://huggingface.co/MahmoodLab/UNI2-h), paper: / 
- model_name: "hoptim_v0", num_features: 1536, weight: [/²,³](https://huggingface.co/bioptimus/H-optimus-0), software
- model_name: "hoptim_v1", num_features: 1536, weight: [/²,³](https://huggingface.co/bioptimus/H-optimus-1), software: [*Hoptimus-1*](https://www.bioptimus.com/h-optimus-1)
- model_name: "ret_ccl", num_features: 2048, weight: [Paip/TCGA pretrained](https://drive.google.com/drive/folders/1AhstAFVqtTqxeS9WlBpU41BV08LYFUnL?usp=sharing) from [official github](https://github.com/Xiyue-Wang/RetCCL) , paper: [*RetCCL: Clustering-guided contrastive learning for whole-slide image retrieval*](https://www.sciencedirect.com/science/article/abs/pii/S1361841522002730)
- model_name: "virchow_v2", num_features: 2560, weight: [/²,³](https://huggingface.co/paige-ai/Virchow2), paper: [*Virchow2: Scaling Self-Supervised Mixed Magnification Models in Pathology*](https://arxiv.org/abs/2408.00738)

/!\ ctranspath demands another version of the timm library 


In [2]:
model_name = "virchow_v2" # TODO: retest: ctranspath (wrong timm)
weight_fold_path = Path("../../../../../weight_fold")
weight = get_weight(model_name, weight_fold_path)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model {model_name} with weight {weight} on device {device}")

Loading model virchow_v2 with weight ../../../../../weight_fold/virchow/v2/weight on device cuda


## Load and encode an image ##

In [3]:
model = Model(model_name, weight, device)

query = Path("../../../../../Data/patches/CrC/CRC-VAL-HE-7K/ADI/ADI-TCGA-AAICEQFN.tif")
query_im = open_and_convert_image(query)
with torch.no_grad():
    vec = model.encode(query_im)
print(vec.shape)

torch.Size([1, 2560])


In [4]:
print(torch.cuda.memory_allocated()/1000000)
del model
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated()/1000000)

2555.041792
8.52992
